# Gold — claims and benefits serving data

Origami Health · synthetic training only. Website: lab2. Planned block: 20 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Build business-facing claim and benefit datasets without multiplying claims when a member has multiple plans.

**Before you begin:** Notebook 02 passed all reconciliation checks. Do not run Gold over an incomplete Silver output.

**Key terms:** Gold is the business-serving layer. Grain means what one row represents. Claim detail has one row per claim; monthly claims have one row per month, provider and service. Fan-out means a join unexpectedly multiplies rows.

**Timebox:** This is the shared 20-minute block for notebooks 03 AND 04, not 20 minutes each. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 2 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell for your slot.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Build Gold outputs without enrollment fan-out

**Purpose:** Build Gold outputs without enrollment fan-out

**Inputs:** Persisted Silver claims and reference dimensions.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Five Gold outputs with exact fixture fingerprints and reconciled counts.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 2:** Run the Gold cell. Locate the groupBy keys in the monthly calculation and explain the row grain. Compare this with enrollment_id as the key of member-plan rows.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
claims=read_delta(spark,cfg,'silver_claims')
monthly=(claims.withColumn('service_month',F.substring('service_date',1,7))
 .groupBy('service_month','provider_id','service_code').agg(F.count('*').alias('claim_count'),F.sum('submitted_centavos').alias('submitted_centavos'),F.sum('paid_centavos').alias('paid_centavos'),F.sum('processing_days').alias('processing_days_sum'))
 .withColumn('currency',F.lit('PHP')).select('service_month','provider_id','service_code','currency','claim_count','submitted_centavos','paid_centavos','processing_days_sum'))
enroll=read_delta(spark,cfg,'silver_enrollments');plans=read_delta(spark,cfg,'silver_plans')
unique(plans,['plan_id','plan_version'])
mp=enroll.join(plans.select('plan_id','plan_version','document_id'),['plan_id','plan_version'],'inner').select(*enroll.columns,'document_id')
assert mp.count()==enroll.count()
tables={'claim_detail':(claims,['claim_id']),'claims_monthly':(monthly,['service_month','provider_id','service_code']),'member_plan':(mp,['enrollment_id']),'benefit_usage':(read_delta(spark,cfg,'silver_benefit_usage'),['usage_id']),'benefit_schedule':(read_delta(spark,cfg,'silver_benefit_schedule'),['benefit_id'])}
for name,(frame,key) in tables.items():
    saved=persist(spark,cfg,'gold_'+name,frame,key);verify_fixture(spark,cfg,'gold_'+name,saved)
assert monthly.agg(F.sum('claim_count')).first()[0]==claims.count()
claims.select('claim_id','submitted_centavos','currency').show(3)
save_evidence(cfg,'gold',{'claims':claims.count(),'monthly_rows':monthly.count(),'no_fanout':True})

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: Five Gold outputs pass fingerprints: claim_detail 720, claims_monthly 36, member_plan 100, benefit_usage 80 and benefit_schedule 6. Monthly claim counts sum back to 720.
- [ ] I saved evidence: Record the five row counts and the monthly reconciliation. Keep the monetary unit explicit. The notebook saves gold.json.

**Discuss:** Why not join every claim to every active plan of its member?

**Self-check explanation:** A member can have two plans. Such a join can double counts and amounts. This exercise retains the billed enrollment on each claim and builds the member-plan lookup separately.

**Next:** day1/04_lakehouse.ipynb. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
